# 🎯 Object Detection using YOLOv8 — BYOP (Computer Vision)

**Course:** Computer Vision  
**Deadline:** Mar 31, 2026  
**Problem:** Real-time object detection on images, videos, and webcam feeds

---

This notebook demonstrates a complete object detection pipeline using **YOLOv8** (You Only Look Once).  
We cover:

- Installing dependencies (in-notebook)
- Running detection on a sample image
- Running detection on a video file
- Visualizing results with bounding boxes & confidence scores
- Saving annotated outputs
- (Optional) Webcam / live feed detection


## 📦 Step 1 — Install Dependencies (No Local pip needed outside notebook)

Run this cell once. It installs everything inside the notebook's kernel environment.


In [1]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

packages = [
    "ultralytics",       # YOLOv8
    "opencv-python",     # Image/video processing
    "matplotlib",        # Visualization
    "Pillow",            # Image I/O
    "requests",          # Download sample images
    "ipywidgets",        # Notebook interactivity
    "tqdm",              # Progress bars
]

for pkg in packages:
    print(f"Installing {pkg}...")
    install(pkg)

print("\n✅ All dependencies installed successfully!")

Installing ultralytics...
Installing opencv-python...
Installing matplotlib...
Installing Pillow...
Installing requests...
Installing ipywidgets...
Installing tqdm...

✅ All dependencies installed successfully!


## 🔧 Step 2 — Imports & Setup


In [2]:
import os
import cv2
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from io import BytesIO
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image as IPImage, Video
import warnings
warnings.filterwarnings('ignore')

# Create output directories
os.makedirs("outputs/images", exist_ok=True)
os.makedirs("outputs/videos", exist_ok=True)
os.makedirs("sample_data", exist_ok=True)

print("✅ Imports successful. Directories created.")
print(f"OpenCV version  : {cv2.__version__}")

import ultralytics
print(f"Ultralytics ver : {ultralytics.__version__}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Arya\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 🤖 Step 3 — Load YOLOv8 Model

YOLOv8 comes in several sizes:  
`yolov8n` (nano, fastest) → `yolov8s` → `yolov8m` → `yolov8l` → `yolov8x` (largest, most accurate)

We use `yolov8n` for speed. The model is auto-downloaded on first run.


In [ ]:
MODEL_NAME = "yolov8n.pt"  # Change to yolov8s.pt, yolov8m.pt etc. for more accuracy

model = YOLO(MODEL_NAME)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Number of classes: {len(model.names)}")
print(f"\nSample classes (first 20):")
for idx, name in list(model.names.items())[:20]:
    print(f"  [{idx:3d}] {name}")

## 🖼️ Step 4 — Detect Objects in a Sample Image

We download a public domain street scene image for demonstration.


In [ ]:
# ── Download a sample image ──────────────────────────────────────────────────
SAMPLE_IMAGE_URL = "https://ultralytics.com/images/bus.jpg"
SAMPLE_IMAGE_PATH = "sample_data/bus.jpg"

if not os.path.exists(SAMPLE_IMAGE_PATH):
    print("Downloading sample image...")
    response = requests.get(SAMPLE_IMAGE_URL)
    with open(SAMPLE_IMAGE_PATH, "wb") as f:
        f.write(response.content)
    print("✅ Downloaded!")
else:
    print("✅ Sample image already exists.")

# Show original image
img = Image.open(SAMPLE_IMAGE_PATH)
plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.title("Original Image", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f"Image size: {img.size[0]}×{img.size[1]} px")

In [ ]:
# ── Run Object Detection ─────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.35  # Minimum confidence to show a detection

results = model.predict(
    source=SAMPLE_IMAGE_PATH,
    conf=CONFIDENCE_THRESHOLD,
    save=False,          # We'll handle saving manually
    verbose=False
)

result = results[0]
boxes   = result.boxes
n_dets  = len(boxes)

print(f"✅ Detection complete!")
print(f"Objects detected: {n_dets}")
print()

# Print each detection
print(f"{'#':>3}  {'Class':<15}  {'Confidence':>10}  {'Bounding Box (x1,y1,x2,y2)'}")
print("-" * 65)
for i, box in enumerate(boxes):
    cls_id = int(box.cls[0])
    label  = model.names[cls_id]
    conf   = float(box.conf[0])
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    print(f"{i+1:>3}  {label:<15}  {conf:>10.2%}  ({x1}, {y1}, {x2}, {y2})")

In [ ]:
# ── Visualize Detections with Custom Bounding Boxes ──────────────────────────
COLORS = [
    "#FF4B4B", "#4BFF91", "#4B9FFF", "#FFD24B",
    "#FF4BCC", "#4BFFFF", "#FF8C4B", "#A04BFF"
]

img_cv = cv2.imread(SAMPLE_IMAGE_PATH)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img_rgb)

for i, box in enumerate(boxes):
    cls_id = int(box.cls[0])
    label  = model.names[cls_id]
    conf   = float(box.conf[0])
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    w, h = x2 - x1, y2 - y1

    color = COLORS[cls_id % len(COLORS)]

    rect = patches.Rectangle(
        (x1, y1), w, h,
        linewidth=2.5,
        edgecolor=color,
        facecolor='none'
    )
    ax.add_patch(rect)

    caption = f"{label} {conf:.0%}"
    ax.text(
        x1, y1 - 6,
        caption,
        color='white',
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none')
    )

ax.set_title(
    f"YOLOv8 Object Detection — {n_dets} object(s) found  |  conf ≥ {CONFIDENCE_THRESHOLD:.0%}",
    fontsize=13, fontweight='bold', pad=12
)
ax.axis('off')
plt.tight_layout()

OUTPUT_IMAGE = "outputs/images/detected_bus.jpg"
plt.savefig(OUTPUT_IMAGE, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Annotated image saved → {OUTPUT_IMAGE}")

## 📊 Step 5 — Detection Statistics & Analysis


In [ ]:
from collections import Counter

class_counts = Counter()
conf_scores  = []

for box in boxes:
    cls_id = int(box.cls[0])
    label  = model.names[cls_id]
    conf   = float(box.conf[0])
    class_counts[label] += 1
    conf_scores.append(conf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Detection Analytics", fontsize=15, fontweight='bold', y=1.02)

# ── Bar chart: object counts ─────────────────────────────────────────────────
ax1 = axes[0]
labels  = list(class_counts.keys())
counts  = list(class_counts.values())
bars = ax1.bar(labels, counts, color=[COLORS[i % len(COLORS)] for i in range(len(labels))],
               edgecolor='black', linewidth=0.8)
ax1.set_title("Objects Detected by Class", fontweight='bold')
ax1.set_xlabel("Class")
ax1.set_ylabel("Count")
ax1.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(count), ha='center', va='bottom', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# ── Histogram: confidence distribution ──────────────────────────────────────
ax2 = axes[1]
ax2.hist(conf_scores, bins=10, range=(0, 1),
         color='#4B9FFF', edgecolor='black', linewidth=0.8, alpha=0.85)
ax2.axvline(np.mean(conf_scores), color='red', linestyle='--',
            linewidth=2, label=f"Mean = {np.mean(conf_scores):.2f}")
ax2.set_title("Confidence Score Distribution", fontweight='bold')
ax2.set_xlabel("Confidence Score")
ax2.set_ylabel("Frequency")
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/images/detection_stats.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Average confidence: {np.mean(conf_scores):.2%}")
print(f"Min confidence    : {np.min(conf_scores):.2%}")
print(f"Max confidence    : {np.max(conf_scores):.2%}")

## 🎞️ Step 6 — Object Detection on a Video File

We download a short sample video and run detection frame-by-frame.


In [ ]:
# ── Download sample video ────────────────────────────────────────────────────
VIDEO_URL  = "https://ultralytics.com/assets/people-walking.mp4"
VIDEO_IN   = "sample_data/sample_video.mp4"
VIDEO_OUT  = "outputs/videos/detected_output.mp4"

if not os.path.exists(VIDEO_IN):
    print("Downloading sample video...")
    r = requests.get(VIDEO_URL, stream=True)
    with open(VIDEO_IN, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("✅ Downloaded!")
else:
    print("✅ Sample video already exists.")

cap = cv2.VideoCapture(VIDEO_IN)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"\nVideo info:")
print(f"  Resolution : {width}×{height}")
print(f"  FPS        : {fps:.1f}")
print(f"  Frames     : {total_frames}")
print(f"  Duration   : {total_frames/fps:.1f}s")

In [ ]:
# ── Process Video — Detection on every frame ──────────────────────────────────
from tqdm.notebook import tqdm

cap    = cv2.VideoCapture(VIDEO_IN)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (width, height))

frame_count  = 0
total_dets   = 0
PROCESS_EVERY = 1  # Process every Nth frame (1 = all frames, 2 = skip alternate)

with tqdm(total=total_frames, desc="Processing video") as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % PROCESS_EVERY == 0:
            results_v = model.predict(
                source=frame,
                conf=CONFIDENCE_THRESHOLD,
                verbose=False
            )
            annotated = results_v[0].plot()  # Built-in annotator
            total_dets += len(results_v[0].boxes)
        else:
            annotated = frame

        out.write(annotated)
        frame_count += 1
        pbar.update(1)

cap.release()
out.release()

print(f"\n✅ Video processing complete!")
print(f"Frames processed : {frame_count}")
print(f"Total detections : {total_dets}")
print(f"Avg detections/frame: {total_dets/frame_count:.1f}")
print(f"Output saved → {VIDEO_OUT}")

## 🖥️ Step 7 — (Optional) Webcam / Live Feed Detection

> ⚠️ **Only run this if you have a webcam connected.**  
> Press **`Q`** in the OpenCV window to quit.
> This will NOT work in remote/cloud notebook environments.


In [ ]:
RUN_WEBCAM = False  # ← Change to True if you have a webcam

if RUN_WEBCAM:
    cap = cv2.VideoCapture(0)  # 0 = default webcam

    if not cap.isOpened():
        print("❌ Could not open webcam. Check connection.")
    else:
        print("✅ Webcam opened. Press 'Q' in the window to quit.")
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results_wc = model.predict(source=frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
            annotated  = results_wc[0].plot()

            cv2.imshow("YOLOv8 — Live Object Detection (Q to quit)", annotated)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        print("✅ Webcam stream closed.")
else:
    print("ℹ️  Webcam detection skipped. Set RUN_WEBCAM = True to enable.")

## 🖼️ Step 8 — Use Your Own Image

Upload any image and change the path below.


In [ ]:
# ── Detect on a custom image ──────────────────────────────────────────────────
# Replace the path below with your own image
CUSTOM_IMAGE = "sample_data/bus.jpg"   # ← Change this to your image path

if os.path.exists(CUSTOM_IMAGE):
    custom_results = model.predict(
        source=CUSTOM_IMAGE,
        conf=CONFIDENCE_THRESHOLD,
        verbose=False
    )

    annotated_img = custom_results[0].plot()  # numpy array (BGR)
    annotated_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(14, 8))
    plt.imshow(annotated_rgb)
    plt.title(f"Custom Image Detection — {len(custom_results[0].boxes)} object(s)",
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()

    OUTPUT_CUSTOM = "outputs/images/custom_detected.jpg"
    plt.savefig(OUTPUT_CUSTOM, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → {OUTPUT_CUSTOM}")
else:
    print(f"❌ File not found: {CUSTOM_IMAGE}")
    print("Upload your image to the sample_data/ folder and update the path above.")

## 📋 Step 9 — Summary & Reflection

### What was built

A complete object detection pipeline using **YOLOv8** that:

- Detects 80 COCO object classes in images and videos
- Visualizes detections with color-coded bounding boxes
- Provides detection statistics (class counts, confidence distribution)
- Saves annotated outputs for submission

### Key Concepts Applied (Computer Vision Course)

| Concept                       | Applied As                            |
| ----------------------------- | ------------------------------------- |
| Convolutional Neural Networks | YOLOv8 backbone (CSPDarknet)          |
| Feature Pyramid Networks      | YOLOv8 neck for multi-scale detection |
| Non-Maximum Suppression (NMS) | Removing duplicate bounding boxes     |
| Anchor-free detection         | YOLOv8's head design                  |
| Bounding Box Regression       | IoU-based loss (CIoU)                 |
| Image preprocessing           | Letterbox resize, normalization       |

### Challenges Faced

- Managing confidence threshold: too low → many false positives; too high → missed objects
- Video processing is memory-intensive — frame-by-frame approach is required
- Webcam access varies across environments (works locally, not on cloud)

### Learnings

- YOLOv8 is extremely fast and accurate for real-time detection tasks
- Pre-trained COCO weights cover most common real-world objects
- Confidence thresholding is a key hyperparameter in deployment
- OpenCV's VideoWriter requires codec compatibility (mp4v works broadly)


In [ ]:
# ── Final output summary ─────────────────────────────────────────────────────
print("="*55)
print("       BYOP — Object Detection Project Summary")
print("="*55)
print(f"  Model         : YOLOv8n (nano)")
print(f"  Classes       : {len(model.names)} (COCO dataset)")
print(f"  Conf threshold: {CONFIDENCE_THRESHOLD:.0%}")
print()
print("  Outputs generated:")
for f in Path("outputs").rglob("*"):
    if f.is_file():
        size_kb = f.stat().st_size // 1024
        print(f"    ✅ {f}  ({size_kb} KB)")
print("="*55)